In [1]:
# Model Evaluation Lab
# Complete the exercises by filling in the code where indicated with TODO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import load_breast_cancer, load_diabetes
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("MODEL EVALUATION LAB - STUDENT VERSION")
print("=" * 80)

# ============================================================================
# PART 1: CLASSIFICATION EVALUATION
# ============================================================================

print("\n" + "=" * 80)
print("PART 1: CLASSIFICATION EVALUATION - BREAST CANCER DATASET")
print("=" * 80)

# Load the breast cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print(f"\nDataset Shape: {X.shape}")
print(f"Number of Features: {X.shape[1]}")
print(f"Number of Samples: {X.shape[0]}")
print(f"\nClass Distribution:")
print(y.value_counts())

# ============================================================================
# Exercise 1.1: Train-Test Split
# Task: Split the data into training (80%) and test (20%) sets
# Use stratified sampling to maintain class distribution
# Set random_state=42 for reproducibility
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.1: Train-Test Split")
print("-" * 80)

# TODO: Split the data using train_test_split
# Hint: Use test_size=0.2, random_state=42, and stratify=y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


# TODO: Print the sizes of training and test sets
# print(f"Training set size: {X_train.shape[0]}")
# print(f"Test set size: {X_test.shape[0]}")

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
# TODO: Standardize the features using StandardScaler
# Remember: fit_transform on training, transform on test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # TODO: fit and transform training data
X_test_scaled =  scaler.transform(X_test)   # TODO: transform test data only

# ============================================================================
# Exercise 1.2: Train and Evaluate Logistic Regression
# Task: Train a logistic regression model and calculate accuracy
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.2: Train and Evaluate Logistic Regression")
print("-" * 80)

# TODO: Create and train a LogisticRegression model
# Hint: Use max_iter=10000, random_state=42
log_reg = LogisticRegression(max_iter=10000, random_state=42)

  # TODO: Create model
# TODO: Fit the model on scaled training data
log_reg.fit(X_train_scaled, y_train)
# TODO: Make predictions on both training and test sets
y_pred_train = log_reg.predict(X_train_scaled)  # TODO: Predict on training set
y_pred_test = log_reg.predict(X_test_scaled)   # TODO: Predict on test set

# TODO: Calculate accuracy for both sets
train_accuracy = accuracy_score(y_train, y_pred_train)  # TODO: Calculate training accuracy
test_accuracy = accuracy_score(y_test, y_pred_test)

# TODO: Print the accuracies and interpret the results
print("Training Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

# Is the model overfitting, underfitting, or well-fitted?
if train_accuracy > test_accuracy + 0.05:
    print("Model is overfitting")
elif test_accuracy > train_accuracy:
    print("Model is underfitting")
else:
    print("Model is well-fitted")

# ============================================================================
# Exercise 1.3: Confusion Matrix Analysis
# Task: Create a confusion matrix and extract TP, TN, FP, FN
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.3: Confusion Matrix Analysis")
print("-" * 80)

# TODO: Create confusion matrix for test predictions
cm = confusion_matrix(y_test, y_pred_test)


# TODO: Print the confusion matrix
# print("\nConfusion Matrix:")
print(cm)

# TODO: Extract TP, TN, FP, FN from confusion matrix
# Hint: Use cm.ravel() or index directly
tn, fp, fn, tp = cm.ravel()


# TODO: Calculate metrics manually using the formulas:
# Accuracy = (TP + TN) / (TP + TN + FP + FN)
# Precision = TP / (TP + FP)
# Recall = TP / (TP + FN)
# F1 Score = 2 * (Precision * Recall) / (Precision + Recall)

manual_accuracy = (tp + tn) / (tp + tn + fp + fn)
manual_precision = tp / (tp + fp)
manual_recall = tp / (tp + fn)
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall)

# TODO: Print the manually calculated metrics
print("Manual Accuracy:", manual_accuracy)
print("Manual Precision:", manual_precision)
print("Manual Recall:", manual_recall)
print("Manual F1 Score:", manual_f1)

# TODO: Visualize the confusion matrix using seaborn heatmap
plt.figure(figsize=(8, 6))
# TODO: Create heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig('confusion_matrix_student.png')
plt.close()

# ============================================================================
# Exercise 1.4: Precision, Recall, and F1 Score
# Task: Calculate metrics using sklearn functions and interpret them
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.4: Precision, Recall, and F1 Score")
print("-" * 80)

# TODO: Calculate metrics using sklearn functions
precision = precision_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

# TODO: Print the metrics with interpretations
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# TODO: Print the full classification report
# Hint: Use classification_report with target_names=['Malignant', 'Benign']
print(classification_report(y_test, y_pred_test, target_names=['Malignant', 'Benign']))


# ============================================================================
# Exercise 1.5: ROC Curve and AUC
# Task: Generate ROC curve and calculate AUC score
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.5: ROC Curve and AUC")
print("-" * 80)

# TODO: Get predicted probabilities for the positive class
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# TODO: Calculate ROC curve coordinates
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# TODO: Calculate AUC score
auc_score = roc_auc_score(y_test, y_pred_proba)

# TODO: Print AUC score and interpret it
print("AUC Score:", auc_score)
# AUC >= 0.9: Excellent, >= 0.8: Good, >= 0.7: Acceptable, < 0.7: Poor
if auc_score >= 0.9:
    print("Model performance: Excellent")
elif auc_score >= 0.8:
    print("Model performance: Good")
elif auc_score >= 0.7:
    print("Model performance: Acceptable")
else:
    print("Model performance: Poor")

# TODO: Plot ROC curve
plt.figure(figsize=(8, 6))
# Plot ROC curve
# Plot diagonal line for random classifier
# Add labels, title, legend
plt.plot(fpr, tpr, label="ROC Curve")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.savefig('roc_curve_student.png')
plt.close()

# ============================================================================
# Exercise 1.6: K-Fold Cross-Validation
# Task: Perform 5-fold stratified cross-validation
# ============================================================================


print("\n" + "-" * 80)
print("Exercise 1.6: K-Fold Cross-Validation")
print("-" * 80)

# TODO: Create StratifiedKFold with 5 splits
# Stratified keeps class balance same in each fold
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: Scale the entire dataset
# We scale full X before cross-validation
scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X)

# TODO: Perform cross-validation using Logistic Regression
# Model will train and test 5 times using different splits
cv_scores = cross_val_score(
    LogisticRegression(max_iter=10000, random_state=42),
    X_scaled_full,
    y,
    cv=kfold,
    scoring='accuracy'
)

# Store mean, std and number of folds
mean = cv_scores.mean()
std = cv_scores.std()
n = len(cv_scores)

# TODO: Print individual fold scores
for i, score in enumerate(cv_scores, 1):
    print("Fold", i, "Accuracy:", score)

# TODO: Print mean and standard deviation
print("Mean CV Accuracy:", mean)
print("Standard Deviation:", std)

# TODO: 95% confidence interval
# Gives range where true accuracy likely exists
lower = mean - 1.96 * (std / np.sqrt(n))
upper = mean + 1.96 * (std / np.sqrt(n))

print("95% Confidence Interval:", lower, "to", upper)

# TODO: Compare with single train-test split accuracy
print("Train-Test Accuracy:", test_accuracy)


# ============================================================================
# Exercise 1.7: Compare Multiple Classifiers
# Task: Train and compare Logistic Regression, Decision Tree, and Random Forest
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 1.7: Compare Multiple Classifiers")
print("-" * 80)

# TODO: Define a dictionary of models
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

# TODO: Loop through each model
for name, model in models.items():
     # TODO: Train the model
    model.fit(X_train_scaled, y_train)

    # TODO: Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]


   # TODO: Calculate all metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    # TODO: Append results to list
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1_val,
        'AUC': auc
    })

# TODO: Create DataFrame and print comparison
results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

# TODO: Identify and print the best model by F1 Score
best_model = results_df.loc[results_df['F1 Score'].idxmax()]
print("\nBest Model based on F1 Score:", best_model['Model'])



# ============================================================================
# PART 2: REGRESSION EVALUATION
# ============================================================================

print("\n" + "=" * 80)
print("PART 2: REGRESSION EVALUATION - DIABETES DATASET")
print("=" * 80)

# Load diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='target')

print(f"\nDataset Shape: {X_reg.shape}")
print(f"\nTarget Statistics:")
print(y_reg.describe())

# ============================================================================
# Exercise 2.1: Train-Test Split for Regression
# Task: Split regression data into training and test sets
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.1: Train-Test Split for Regression")
print("-" * 80)

# TODO: Split the regression data (80-20 split)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# TODO: Print set sizes
print("Train size:", X_train_reg.shape[0])
print("Test size:", X_test_reg.shape[0])

# TODO: Standardize features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)


# ============================================================================
# Exercise 2.2: Train Linear Regression and Calculate Metrics
# Task: Train linear regression and calculate MAE, MSE, RMSE, R²
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.2: Train Linear Regression and Calculate Metrics")
print("-" * 80)
# TODO: Create and train LinearRegression model
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg_scaled, y_train_reg)

# TODO: Make predictions on both sets
y_pred_train_reg = lin_reg.predict(X_train_reg_scaled)
y_pred_test_reg = lin_reg.predict(X_test_reg_scaled)

# TODO: Calculate MAE for both sets
train_mae = mean_absolute_error(y_train_reg, y_pred_train_reg)
test_mae = mean_absolute_error(y_test_reg, y_pred_test_reg)

# TODO: Calculate MSE for both sets
train_mse = mean_squared_error(y_train_reg, y_pred_train_reg)
test_mse = mean_squared_error(y_test_reg, y_pred_test_reg)

# TODO: Calculate RMSE
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

# TODO: Calculate R² score
train_r2 = r2_score(y_train_reg, y_pred_train_reg)
test_r2 = r2_score(y_test_reg, y_pred_test_reg)

# TODO: Print all metrics
print("Train MAE:", train_mae)
print("Test MAE:", test_mae)
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)
print("Train R2:", train_r2)
print("Test R2:", test_r2)




# For example: "Average prediction error is X units"
print("Average prediction error (Test MAE):", test_mae, "units")

# "Model explains X% of variance"
print("Model explains", test_r2 * 100, "% of variance in the data")

# ============================================================================
# Exercise 2.3: Visualize Predictions vs Actual
# Task: Create scatter plots comparing predictions to actual values
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.3: Visualize Predictions vs Actual")
print("-" * 80)

# TODO: Create figure with 2 subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Plot training set (left subplot)
axes[0].scatter(y_train_reg, y_pred_train_reg)
axes[0].plot([y_train_reg.min(), y_train_reg.max()],
             [y_train_reg.min(), y_train_reg.max()])

axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Training Data: Actual vs Predicted")

# TODO: Plot test set (right subplot)
axes[1].scatter(y_test_reg, y_pred_test_reg)
axes[1].plot([y_test_reg.min(), y_test_reg.max()],
             [y_test_reg.min(), y_test_reg.max()])
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].set_title("Test Data: Actual vs Predicted")

# TODO: Save the figure
plt.savefig('regression_predictions_student.png')
plt.close()





# ============================================================================
# Exercise 2.4: Residual Analysis
# Task: Calculate and visualize residuals
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.4: Residual Analysis")
print("-" * 80)
# TODO: Calculate residuals (actual - predicted)
residuals = y_test_reg - y_pred_test_reg

# TODO: Print residual statistics
print("Residual Mean:", residuals.mean())
print("Residual Std:", residuals.std())
print("Residual Min:", residuals.min())
print("Residual Max:", residuals.max())

# TODO: Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Left subplot - Residual plot
axes[0].scatter(y_pred_test_reg, residuals)
axes[0].axhline(y=0)
axes[0].set_title("Residual Plot")

# TODO: Right subplot - Residual distribution
axes[1].hist(residuals, bins=30)
axes[1].axvline(x=0)
axes[1].set_title("Residual Distribution")

# TODO: Save the figure
plt.savefig('residual_analysis_student.png')
plt.close()





# ============================================================================
# Exercise 2.5: Compare Multiple Regression Models
# Task: Compare Linear Regression, Ridge (two alpha values), and Decision Tree
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.5: Compare Multiple Regression Models")
print("-" * 80)


# TODO: Define dictionary of regression models
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Ridge (alpha=10.0)': Ridge(alpha=10.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=5)
}

reg_results = []

# TODO: Loop through each model
for name, model in reg_models.items():

    # TODO: Train the model
    model.fit(X_train_reg_scaled, y_train_reg)

    # TODO: Make predictions on both sets
    y_pred_train = model.predict(X_train_reg_scaled)
    y_pred_test = model.predict(X_test_reg_scaled)

    # TODO: Calculate metrics
    train_r2_val = r2_score(y_train_reg, y_pred_train)
    test_r2_val = r2_score(y_test_reg, y_pred_test)
    test_mae_val = mean_absolute_error(y_test_reg, y_pred_test)
    test_rmse_val = np.sqrt(mean_squared_error(y_test_reg, y_pred_test))

    # TODO: Calculate overfitting gap (train R² - test R²)
    overfit_gap = train_r2_val - test_r2_val

    # TODO: Append results
    reg_results.append({
        'Model': name,
        'Train R²': train_r2_val,
        'Test R²': test_r2_val,
        'MAE': test_mae_val,
        'RMSE': test_rmse_val,
        'Overfit Gap': overfit_gap
    })

# TODO: Create DataFrame and print comparison
reg_results_df = pd.DataFrame(reg_results)
print("\nRegression Model Comparison:")
print(reg_results_df.to_string(index=False))

# TODO: Identify and print best model by Test R²
best_model = reg_results_df.loc[reg_results_df['Test R²'].idxmax()]
print("\nBest Model based on Test R²:")
print(best_model)


# ============================================================================
# Exercise 2.6: Cross-Validation for Regression
# Task: Perform 5-fold cross-validation for regression
# ============================================================================

print("\n" + "-" * 80)
print("Exercise 2.6: Cross-Validation for Regression")
print("-" * 80)

# TODO: Scale full dataset
scaler_full = StandardScaler()
X_reg_scaled_full = scaler_full.fit_transform(X_reg)

# TODO: Perform cross-validation for R² score
cv_r2_scores = cross_val_score(LinearRegression(), X_reg_scaled_full, y_reg, cv=5, scoring='r2')

# TODO: Perform cross-validation for MAE
# Note: scoring should be 'neg_mean_absolute_error'
# Remember to negate the scores to get positive MAE values
cv_mae_scores = cross_val_score(LinearRegression(), X_reg_scaled_full, y_reg, cv=5, scoring='neg_mean_absolute_error')
cv_mae_scores = -cv_mae_scores

# TODO: Print R² scores for each fold
print("\nCross-Validation R² scores:")
for i, score in enumerate(cv_r2_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print("Mean R²:", np.mean(cv_r2_scores))
print("Std R²:", np.std(cv_r2_scores))

# TODO: Print MAE scores for each fold
print("\nCross-Validation MAE scores:")
for i, score in enumerate(cv_mae_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print("Mean MAE:", np.mean(cv_mae_scores))
print("Std MAE:", np.std(cv_mae_scores))


# ============================================================================
# PART 3: OVERFITTING AND UNDERFITTING DEMONSTRATION
# ============================================================================

print("\n" + "=" * 80)
print("PART 3: OVERFITTING AND UNDERFITTING DEMONSTRATION")
print("=" * 80)

# TODO: Test Decision Trees with different depths: [1, 3, 5, 10, 20]
depths = [1, 3, 5, 10, 20]
train_scores = []
test_scores = []

print("\nDecision Tree Depth Analysis:")
print(f"{'Depth':<10}{'Train R²':<15}{'Test R²':<15}{'Gap':<15}{'Status'}")
print("-" * 70)

# TODO: Loop through each depth
for depth in depths:
    # TODO: Create DecisionTreeRegressor with this depth
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)

    # TODO: Train on scaled training data
    tree.fit(X_train_reg_scaled, y_train_reg)

    # TODO: Calculate R² scores for both sets
    train_score = tree.score(X_train_reg_scaled, y_train_reg)
    test_score = tree.score(X_test_reg_scaled, y_test_reg)

    # TODO: Calculate gap between train and test
    gap = train_score - test_score

    # TODO: Append to lists
    train_scores.append(train_score)
    test_scores.append(test_score)

    # TODO: Determine status
    # If gap > 0.3: "Overfitting"
    # If test_score < 0.3: "Underfitting"
    # Else: "Good Fit"
    if gap > 0.3:
        status = "Overfitting"
    elif test_score < 0.3:
        status = "Underfitting"
    else:
        status = "Good Fit"

    # TODO: Print results
    print(f"{depth:<10}{train_score:<15.4f}{test_score:<15.4f}{gap:<15.4f}{status}")


# TODO: Plot learning curves
plt.figure(figsize=(10, 6))

plt.plot(depths, train_scores, marker='o', label='Training R²')
plt.plot(depths, test_scores, marker='o', label='Test R²')

plt.xlabel("Tree Depth")
plt.ylabel("R² Score")
plt.title("Overfitting vs Underfitting Analysis")
plt.legend()

plt.savefig('overfitting_analysis_student.png')
plt.close()


# ============================================================================
# ASSESSMENT QUESTIONS
# Answer these questions in comments after completing the exercises
# ============================================================================

"""
ASSESSMENT QUESTIONS:

1. What is the difference between accuracy and F1 score? When would you prefer F1 score over accuracy?

Accuracy means total correct predictions out of all.
F1 score means balance between precision and recall.
We use F1 when data is unequal or when both types of mistakes are important.

2. Explain the difference between precision and recall. Give an example of when you would optimize for high precision vs high recall.

Precision means when model predicts YES, how many are actually YES.
Recall means out of all real YES cases, how many model finds.
Example: spam email → precision important.
Cancer detection → recall important.

3. What does an AUC score of 0.5 indicate? What about 1.0?

AUC 0.5 means model is just guessing randomly.
AUC 1.0 means model is perfect.

4. Why do we use cross-validation instead of a single train-test split?

Cross validation checks model multiple times using different data splits.
So result is more accurate and reliable.

5. What is the difference between MAE and RMSE? When would each be preferred?

MAE gives average error value.
RMSE gives more weight to big errors.
If large mistakes matter more, RMSE is preferred.

6. What does an R² score of 0.7 mean? What about a negative R² score?

R² = 0.7 means model explains 70% of data variation.
Negative R² means model is performing very badly.

7. How can you identify overfitting from looking at training and test scores? What are some strategies to reduce overfitting?

If training score high and test score low → overfitting.
To reduce:
use simpler model, more data, regularization, pruning.

8. In the overfitting analysis, which tree depth showed the best balance between training and test performance? Why?

Depth 3 or 5 is better because training and test scores are balanced.
Model is learning properly without overfitting.

9. Why is it important to use stratified sampling for classification problems?

Stratified sampling keeps class ratio same in train and test.
So model learns correctly from both classes.

10. What is the purpose of residual analysis in regression? What should we look for in a residual plot?

Residual analysis shows prediction errors.
Residual plot should be random around zero.
If pattern appears, model is not performing well.


"""

# ============================================================================
# SUBMISSION CHECKLIST
# ============================================================================

"""
Before submitting, ensure you have:
Completed all TODO items
Generated all required visualizations
Answered all assessment questions
Tested your code and verified it runs without errors
Interpreted your results with comments
Compared model performances
Analyzed overfitting/underfitting

Expected output files:
- confusion_matrix_student.png
- roc_curve_student.png
- regression_predictions_student.png
- residual_analysis_student.png
- overfitting_analysis_student.png
"""

MODEL EVALUATION LAB - STUDENT VERSION

PART 1: CLASSIFICATION EVALUATION - BREAST CANCER DATASET

Dataset Shape: (569, 30)
Number of Features: 30
Number of Samples: 569

Class Distribution:
target
1    357
0    212
Name: count, dtype: int64

--------------------------------------------------------------------------------
Exercise 1.1: Train-Test Split
--------------------------------------------------------------------------------
Train size: 455
Test size: 114

--------------------------------------------------------------------------------
Exercise 1.2: Train and Evaluate Logistic Regression
--------------------------------------------------------------------------------
Training Accuracy: 0.989010989010989
Test Accuracy: 0.9824561403508771
Model is well-fitted

--------------------------------------------------------------------------------
Exercise 1.3: Confusion Matrix Analysis
--------------------------------------------------------------------------------
[[41  1]
 [ 1 71]]
Man

'\nBefore submitting, ensure you have:\nCompleted all TODO items\nGenerated all required visualizations\nAnswered all assessment questions\nTested your code and verified it runs without errors\nInterpreted your results with comments\nCompared model performances\nAnalyzed overfitting/underfitting\n\nExpected output files:\n- confusion_matrix_student.png\n- roc_curve_student.png\n- regression_predictions_student.png\n- residual_analysis_student.png\n- overfitting_analysis_student.png\n'